In [1]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [2]:
import os
import sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# os.environ["MKL_THREADING_LAYER"] = "GNU"

sys.path.append("../02-encoding")
sys.path.append(os.path.join(os.environ["NB2P_PREFIX"], "nb2p", "dfgtree"))

In [3]:
from pkgimp import *

from nb2p import fileop, npop, config, stmodel, database

/home/haotian/r/ascent/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
TRAIN_DATASET_NAME = "distilkaggle"
TEST_DATASET_NAME = "distilkaggle"

In [5]:
TRAIN_DIRS = config.dirs(dataset_name=TRAIN_DATASET_NAME)
TRAIN_DIRS.makedirs()
TEST_DIRS = config.dirs(dataset_name=TEST_DATASET_NAME)
TEST_DIRS.makedirs()

making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-ast
making dirs: /ssd/haotian/scs/distilkaggle
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-full
making dirs: /ssd/haotian/scs/distilkaggle/dfgtree
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-eda
making dirs: /ssd/haotian/scs/distilkaggle/logs/full
making dirs: /ssd/haotian/scs/distilkaggle/models/full
making dirs: /ssd/haotian/scs/distilkaggle/ipynb
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-ast
making dirs: /ssd/haotian/scs/distilkaggle
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-full
making dirs: /ssd/haotian/scs/distilkaggle/dfgtree
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-eda
making dirs: /ssd/haotian/scs/distilkaggle/logs/full
making dirs: /ssd/haotian/scs/distilkaggle/models/full
making dirs: /ssd/haotian/scs/distilkaggle/ipynb


## Load Data & Pre-processing

### Load dataset

In [6]:
MAX_LENGTH = 256
MAX_LENGTH

256

Get the list of all samples by globbing the folder

In [7]:
# # for NB2P w/o DTE, use

# train_samples = glob.glob(str(TEST_DIRS.dfgtree / "train-*.lz4"))
# test_samples = glob.glob(str(TEST_DIRS.dfgtree / "test-*.lz4"))

In [8]:
# for others, use

train_samples = sorted(glob.glob(str(TRAIN_DIRS.dfgtree / "train-*.lz4")))
test_samples = sorted(glob.glob(str(TEST_DIRS.dfgtree / "test-*.lz4")))
len(train_samples), len(test_samples)

(234050, 58513)

In [9]:
db, client = database.connect(dataset_name=TRAIN_DATASET_NAME, verbose=True)

Pinged to database nb2p-dk. You successfully connected to MongoDB!


Build List of X-y dicts

In [10]:
ids = set(map(
    lambda x: str(x["_id"]),
    db.notebooksegments.find(
        {"prompted": True, "segment_ends.3": {"$exists": True}, "n_ast_children_of_segments": {"$lte": 256}},
        # {"n_ast_children_of_segments": {"$lt": MAX_LENGTH}, "selected": True},
        {"_id": 1},
    ),
))
len(ids)

285874

Build List of X-y dicts

In [11]:
def clean_segment_ends(segment_ends: List[int]):
    result = set()
    for x in segment_ends:
        if x >= MAX_LENGTH:
            return None, f"segment ends exceed max length: {x}"
        if x not in result and x >= 0:
            result.add(x)

    result = sorted(result)

    if len(result) == 0:
        return None, "empty segment ends after cleaning"

    return result, None


def build_shallow_dataset(samples: List[str]):
    for sample in tqdm(samples):
        # read data
        nb_id = sample.split("/")[-1].split(".")[-2].split("-")[-2]
        if nb_id not in ids:
            # print(f"WARN  ignore {sample}. Reason: should not be included")
            continue

        sample_dict = fileop.read_lz4(sample)
        sample_dict["segment_ends"], err = clean_segment_ends(
            sample_dict["segment_ends"]
        )
        if err:
            # print(f"WARN  ignore {sample}. Reason: {err}")
            continue

        sample_dict["y"] = npop.indices_to_binary(
            sample_dict["segment_ends"], sample_dict["segment_ends"][-1] + 1
        )

        func_encodings = []
        for fdef in sample_dict["func_defs"]:
            func_encodings.append(fdef["repr"])

        encodings = []
        for s in sample_dict["segments"]:
            encodings.extend([c.repr[0] for c in s["repr"].children])

        if len(encodings) == 0:
            # print(f"WARN  ignore {sample}. Reason: encodings is empty")
            continue

        sample_dict["x"] = np.array(encodings)

        code = "\n".join([s['code'] for s in sample_dict['segments']])

        del sample_dict["segments"]
        del sample_dict["func_defs"]
        
        sample_dict["gt_ast"] = sample_dict["segment_ends"]
        sample_dict['code'] = code

        yield sample_dict

In [12]:
train_encodings = list(build_shallow_dataset(train_samples))

100%|█████████████████████████████████████████████████████████| 234050/234050 [01:42<00:00, 2294.09it/s]


In [13]:
len(train_encodings), train_encodings[0]["x"].shape

(228578, (9, 96))

In [14]:
test_encodings = list(build_shallow_dataset(test_samples))

100%|███████████████████████████████████████████████████████████| 58513/58513 [00:23<00:00, 2537.57it/s]


In [15]:
len(test_encodings), test_encodings[0]["x"].shape

(57187, (9, 96))

In [16]:
train_encodings[0]

{'segment_ends': [1, 2, 3, 5, 6, 8],
 'y': array([0, 1, 1, 1, 0, 1, 1, 0, 1]),
 'x': array([[ 9.45315778e-01,  7.56817043e-01,  1.06162119e+00,
         -6.58697262e-02,  3.12040802e-02, -1.61710486e-01,
          5.00966489e-01,  9.96984899e-01, -1.54151097e-01,
          2.00146303e-01, -2.49062538e-01, -1.15335906e+00,
         -1.44633174e+00,  6.41483128e-01,  1.50784826e+00,
         -1.48321116e+00,  2.33889294e+00,  1.34858453e+00,
         -9.67070833e-02,  3.44295263e-01, -6.74885571e-01,
          4.68245357e-01, -2.48466301e+00, -3.76321852e-01,
          8.80073071e-01,  1.41345632e+00, -1.12538576e+00,
          3.01725477e-01,  3.97411346e-01,  7.00807452e-01,
          1.01573181e+00, -1.15354516e-01, -1.13244615e-01,
         -3.94261897e-01, -9.28648040e-02, -6.96923435e-01,
         -7.26813078e-01, -8.02821591e-02,  4.28244412e-01,
         -8.54992867e-01,  4.33234662e-01, -1.87107638e-01,
          3.77800608e+00, -1.13881719e+00, -4.97925848e-01,
         -5.6997

In [17]:
TRAIN_SETUP_NAME = "astn4_256"
TEST_SETUP_NAME = (
    "astn4_256_cross" if TRAIN_DATASET_NAME != TEST_DATASET_NAME else TRAIN_SETUP_NAME
)
TRAIN_SETUP_NAME, TEST_SETUP_NAME

('astn4_256', 'astn4_256')

### Pad to max length

In [18]:
display(test_encodings[0]["x"].shape, test_encodings[0]["x"].shape)
display(
    npop.pad(test_encodings[0]["x"], MAX_LENGTH).shape,
    npop.pad(test_encodings[0]["x"], MAX_LENGTH).shape,
)

(9, 96)

(9, 96)

(256, 96)

(256, 96)

In [19]:
X_train_padded = np.zeros((len(train_encodings), MAX_LENGTH, 96), dtype=np.float32)
y_train_padded = np.zeros((len(train_encodings), MAX_LENGTH), dtype=np.float32)

In [20]:
X_test_padded = np.zeros((len(test_encodings), MAX_LENGTH, 96), dtype=np.float32)
y_test_padded = np.zeros((len(test_encodings), MAX_LENGTH), dtype=np.float32)

In [21]:
### NOTE: An interesting finding that applying multithreading is as expected on parallelism
### but not multiprocessing, as NumPy can only use single CPU core across processes.
MAX_WORKERS = 32


def assign_data4pool(dest, data):
    i, arr = data
    if len(dest.shape) == 2:
        dest[i, : arr.shape[0]] = arr
    elif len(dest.shape) == 3:
        dest[i, : arr.shape[0], :] = arr


def mask_data4pool(data):
    return npop.mask(data["y"].shape[0], MAX_LENGTH)

In [22]:
from functools import partial

X_train_padded_arr = thread_map(
    partial(assign_data4pool, X_train_padded),
    enumerate(d["x"] for d in train_encodings),
    max_workers=MAX_WORKERS,
)
y_train_padded_arr = thread_map(
    partial(assign_data4pool, y_train_padded),
    enumerate(d["y"] for d in train_encodings),
    max_workers=MAX_WORKERS,
)
print(len(X_train_padded_arr), len(y_train_padded_arr))

228578it [00:00, 315419.24it/s]
228578it [00:00, 520540.14it/s]

228578 228578


In [23]:
train_mask = thread_map(mask_data4pool, train_encodings, max_workers=MAX_WORKERS)
len(train_mask), train_mask[0]

100%|███████████████████████████████████████████████████████| 228578/228578 [00:00<00:00, 564081.34it/s]


(228578,
 array([False, False, False, False, False, False, False, False, False,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  Tru

In [24]:
from functools import partial

X_test_padded_arr = thread_map(
    partial(assign_data4pool, X_test_padded),
    enumerate(d["x"] for d in test_encodings),
    max_workers=MAX_WORKERS,
)
y_test_padded_arr = thread_map(
    partial(assign_data4pool, y_test_padded),
    enumerate(d["y"] for d in test_encodings),
    max_workers=MAX_WORKERS,
)
print(len(X_test_padded_arr), len(y_test_padded_arr))

57187it [00:00, 163868.12it/s]
57187it [00:00, 490327.06it/s]

57187 57187


In [25]:
test_mask = thread_map(mask_data4pool, test_encodings, max_workers=MAX_WORKERS)
print(test_mask[0])

100%|█████████████████████████████████████████████████████████| 57187/57187 [00:00<00:00, 398950.92it/s]

[False False False False False False False False False  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  T

In [26]:
train_lengths = list(map(lambda x: (~x).sum(), train_mask))
test_lengths = list(map(lambda x: (~x).sum(), test_mask))
print(len(train_lengths), len(test_lengths), len(train_lengths) + len(test_lengths))
print(train_lengths[0], test_lengths[0])

228578 57187 285765
9 9


In [27]:
DATA_X = X_test_padded
DATA_Y = y_test_padded

In [28]:
import gc
import ctypes

del gc.garbage[:]
gc.collect()
libc = ctypes.CDLL("libc.so.6")
libc.malloc_trim(0)

1

In [29]:
CODE_TRAIN = [e['code'] for e in train_encodings]
CODE_TEST = [e['code'] for e in test_encodings]

## Decision Tree

In [30]:
import nb2p.stmodel.dtree as dtree

config = dtree.DTreeConfig(max_depth=64)

dtree.train_test(
    train_data=(X_train_padded, y_train_padded),
    test_data=(DATA_X, DATA_Y),
    test_lengths=test_lengths,
    code=(CODE_TRAIN, CODE_TEST),
    config=config,
    log_path=dtree.default_log_path(TEST_DIRS, TEST_SETUP_NAME, config),
    model_path=dtree.default_model_path(TRAIN_DIRS, TRAIN_SETUP_NAME, config),
    trained=False,
)[0]

[Parallel(n_jobs=80)]: Using backend ThreadingBackend with 80 concurrent workers.


building tree 1 of 1


[Parallel(n_jobs=80)]: Done   1 tasks      | elapsed:  4.5min


[{"random_state": 42, "n_estimators": 1, "n_jobs": 80, "verbose": 10, "max_depth": 64, "min_samples_split": 0.001}] model dumped to: /ssd/haotian/scs/distilkaggle/models/full/astn4_256-dtree-depth_64-minsplit_0.001.joblib


[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.3s finished


## Random Forest

In [31]:
import nb2p.stmodel.rforest as rforest

config = rforest.RForestConfig(max_depth=64)

rforest.train_test(
    train_data=(X_train_padded, y_train_padded),
    test_data=(DATA_X, DATA_Y),
    test_lengths=test_lengths,
    code=(CODE_TRAIN, CODE_TEST),
    config=config,
    log_path=rforest.default_log_path(TEST_DIRS, TEST_SETUP_NAME, config),
    model_path=rforest.default_model_path(TRAIN_DIRS, TRAIN_SETUP_NAME, config),
    trained=False,
)[0]

[Parallel(n_jobs=100)]: Using backend ThreadingBackend with 100 concurrent workers.


building tree 1 of 100
building tree 2 of 100
building tree 3 of 100
building tree 4 of 100
building tree 5 of 100
building tree 6 of 100
building tree 7 of 100
building tree 8 of 100
building tree 9 of 100
building tree 10 of 100
building tree 11 of 100
building tree 12 of 100
building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100
building tree 17 of 100
building tree 18 of 100
building tree 19 of 100
building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
building tree 32 of 100
building tree 33 of 100
building tree 34 of 100
building tree 35 of 100
building tree 36 of 100
building tree 37 of 100
building tree 38 of 100
building tree 39 of 100
building tree 40 of 100
building tree 41 of 100
building tree 42 of 100
b

[Parallel(n_jobs=100)]: Done  11 out of 100 | elapsed: 15.2min remaining: 122.9min
[Parallel(n_jobs=100)]: Done  22 out of 100 | elapsed: 15.8min remaining: 55.9min
[Parallel(n_jobs=100)]: Done  33 out of 100 | elapsed: 16.0min remaining: 32.4min
[Parallel(n_jobs=100)]: Done  44 out of 100 | elapsed: 16.1min remaining: 20.5min
[Parallel(n_jobs=100)]: Done  55 out of 100 | elapsed: 16.3min remaining: 13.4min
[Parallel(n_jobs=100)]: Done  66 out of 100 | elapsed: 16.5min remaining:  8.5min
[Parallel(n_jobs=100)]: Done  77 out of 100 | elapsed: 16.7min remaining:  5.0min
[Parallel(n_jobs=100)]: Done  88 out of 100 | elapsed: 16.9min remaining:  2.3min
[Parallel(n_jobs=100)]: Done 100 out of 100 | elapsed: 17.2min finished


[{"random_state": 42, "n_estimators": 100, "n_jobs": 100, "verbose": 10, "max_depth": 64, "min_samples_split": 0.001}] model dumped to: /ssd/haotian/scs/distilkaggle/models/full/astn4_256-rforest-n_100-depth_64-minsplit_0.001.joblib


[Parallel(n_jobs=100)]: Using backend ThreadingBackend with 100 concurrent workers.
[Parallel(n_jobs=100)]: Done  11 out of 100 | elapsed:    3.3s remaining:   26.9s
[Parallel(n_jobs=100)]: Done  22 out of 100 | elapsed:    6.2s remaining:   21.8s
[Parallel(n_jobs=100)]: Done  33 out of 100 | elapsed:    8.9s remaining:   18.2s
[Parallel(n_jobs=100)]: Done  44 out of 100 | elapsed:   11.6s remaining:   14.8s
[Parallel(n_jobs=100)]: Done  55 out of 100 | elapsed:   14.3s remaining:   11.7s
[Parallel(n_jobs=100)]: Done  66 out of 100 | elapsed:   17.1s remaining:    8.8s
[Parallel(n_jobs=100)]: Done  77 out of 100 | elapsed:   19.8s remaining:    5.9s
[Parallel(n_jobs=100)]: Done  88 out of 100 | elapsed:   22.5s remaining:    3.1s
[Parallel(n_jobs=100)]: Done 100 out of 100 | elapsed:   25.5s finished


## XGBoost

In [ ]:
import nb2p.stmodel.xgboost as xgboost

config = xgboost.XGBoostConfig(max_depth=8)

xgboost.train_test(
    train_data=(X_train_padded, y_train_padded),
    test_data=(DATA_X, DATA_Y),
    test_lengths=test_lengths,
    code=(CODE_TRAIN, CODE_TEST),
    config=config,
    log_path=xgboost.default_log_path(TEST_DIRS, TEST_SETUP_NAME, config),
    model_path=xgboost.default_model_path(TRAIN_DIRS, TRAIN_SETUP_NAME, config),
    trained=False,
)[0]

[01:16:15] DEBUG: /workspace/src/gbm/gbtree.cc:130: Using tree method: 0


/home/haotian/r/ascent/.venv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [01:16:15] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "num_boost_round" } are not used.

  warnings.warn(smsg, UserWarning)


## Transformer

In [34]:
from nb2p.stmodel.dnn import DNNInput

In [35]:
def seed_everything(seed: int):
    import random, os
    import numpy as np
    import torch
    
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
seed_everything(42)

In [36]:
import torch

input_size = max(s.shape[0] for s in X_test_padded)
eval_freq = 1
checkpoint_freq = 10

print(
    input_size,
    eval_freq,
    checkpoint_freq,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

256 1 10
cuda


In [37]:
from nb2p.stmodel.dnn import Transformer, train

MODEL_NAME = "transformer"

num_heads = 8
hidden_size = 512
num_layers = 4

model = Transformer(
    input_size, num_heads=num_heads, hidden_size=hidden_size, num_layers=num_layers
).to(device)

In [ ]:
train(
    model,
    DNNInput(X=X_train_padded, y=y_train_padded, code=CODE_TRAIN, mask=train_mask),
    DNNInput(X=X_test_padded, y=y_test_padded, code=CODE_TEST, mask=test_mask),
    num_epochs=50,
    batch_size=128,
    eval_freq=eval_freq,
    checkpoint_freq=checkpoint_freq,
    learning_rate=1e-4,
    device=device,
    model_write_dir=TEST_DIRS.log,
    report_interval=2000,
    model_name=(
        f"{TRAIN_SETUP_NAME}-{MODEL_NAME}_bf_ff{hidden_size}_l{num_layers}"
    ),
    # n_try=32,
    # start=0,
)

In [39]:
%xdel model
torch.cuda.empty_cache()
gc.collect()
import ctypes

libc = ctypes.CDLL("libc.so.6")
libc.malloc_trim(0)

1

## NB2P w/o DTE

In [40]:
def seed_everything(seed: int):
    import random, os
    import numpy as np
    import torch
    
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
seed_everything(42)

In [41]:
import torch

input_size = max(s.shape[0] for s in X_test_padded)
eval_freq = 1
checkpoint_freq = 1

print(
    input_size,
    eval_freq,
    checkpoint_freq,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

256 1 1
cuda


In [42]:
from nb2p.stmodel.dnn import BiLSTM, Trainer, DNNInput

MODEL_NAME = "bilstm"

hidden_size = 512
num_layers = 4

model = BiLSTM(input_size, hidden_size=hidden_size, num_layers=num_layers).to(device)

In [ ]:
Trainer(
    model,
    # num_epochs=11, # For DistilKaggle
    num_epochs=7,  # For PMBF
    batch_size=128,
    eval_freq=eval_freq,
    checkpoint_freq=checkpoint_freq,
    learning_rate=1e-3,
    device=device,
    model_write_dir=TEST_DIRS.log,
    report_interval=100,
    window_size=1000,
    model_name=(
        f"{TRAIN_SETUP_NAME}-{MODEL_NAME}_bf_ff{hidden_size}_l{num_layers}"
    ),
    setup='bce',
).train(
    DNNInput(X=X_train_padded, y=y_train_padded, code=CODE_TRAIN, mask=train_mask),
    DNNInput(X=X_test_padded, y=y_test_padded, code=CODE_TEST, mask=test_mask),
    # n_try=32,
)